# Applying SQD to PPP Hamiltonian for Polyacene: Problem Set

This notebook is the **problem-set** of SQD exercise set.
Refer to the tutorial if you have not gone through it yet.

- The **base setup** below uses the **linear** polyacene example with `n_rings = 2`.
- **Problem 2** compares SQD in the original **site basis** and the **RHF-rotated basis**, following the basis-comparison logic introduced in the tutorial.
- **Problem 4** keeps the same linear-geometry PPP utility and treats `n_rings = 3` as a **real-backend target** rather than a local Statevector-sampling task.


In [ ]:
import numpy as np

from polyacene_ppp import build_polyacene_ppp, spinorb_ppp_to_spatial, plot_coordinate, plot_orbital_spread, plot_site_population
from ccsd_wrapper import run_ccsd_from_hg, transform_integrals_to_mo

from pyscf.fci import direct_spin1

### Base setup: linear polyacene with `n_rings = 2`

We first build the **linear** `n_rings = 2` PPP Hamiltonian, convert it to spatial-orbital tensors, and run RHF/CCSD.
These quantities are reused in Problems 1, 2, and 4.

In [ ]:
# PPP model parameters for the base problem-set case: linear n_rings = 2
n_rings = 2

def prepare_case(case_label, n_rings,
                 t0=-2.4, U=11.26, bond_length=1.4, kappa=1.0, run_fci=False):
    """Build the linear PPP model and return the site-basis Hamiltonian data."""

    ppp_spin = build_polyacene_ppp(
        n_rings=n_rings,
        t0=t0,
        U=U,
        bond_length=bond_length,
        kappa=kappa,
    )
    hcore_site, eri_site, energy_constant = spinorb_ppp_to_spatial(ppp_spin, U=U)
    num_orbitals = hcore_site.shape[0]
    nelec = (num_orbitals // 2, num_orbitals // 2)

    return {
        "case": case_label,
        "n_rings": n_rings,
        "ppp_spin": ppp_spin,
        "hcore_site": hcore_site,
        "eri_site": eri_site,
        "energy_constant": energy_constant,
        "num_orbitals": num_orbitals,
        "nelec": nelec,
    }

ppp_n2 = prepare_case("n_ring=2", n_rings=2)

print(f"Polyacene rings: {n_rings}")
print(f"Spatial orbitals: {ppp_n2['num_orbitals']}")
print(f"Spin orbitals / qubits: {2 * ppp_n2['num_orbitals']}")
print(f"Electrons (alpha, beta): {ppp_n2['nelec']}")
print(f"Constant intersite Coulomb shift: {ppp_n2['energy_constant']:.6f} eV")
plot_coordinate(ppp_n2["ppp_spin"])


### Run RHF and CCSD calculation


Next we run RHF/CCSD. The custom driver internally constructs an RHF reference from the supplied `hcore` and `eri` in the site basis, then solves CCSD on top of that reference. This gives us:

- the RHF energy,
- the CCSD energy,
- the CCSD amplitudes `t1` and `t2`, **(will be required for LUCJ ansatz initialization in the main problem)**
- the RHF orbital coefficient matrix `C`.

We then use `C` to rotate the Hamiltonian into the RHF orbital basis. Since the rotation is unitary, the exact Hamiltonian is unchanged; only its representation changes.


In [ ]:
def classical_preprocessing(ppp_case, run_fci=False):
    hcore_site, eri_site = ppp_case["hcore_site"], ppp_case["eri_site"]
    nelec, num_orbitals, energy_constant = ppp_case["nelec"], ppp_case["num_orbitals"], ppp_case["energy_constant"]

    ccsd_result = run_ccsd_from_hg(
        h=hcore_site,
        g=eri_site,
        nelec=sum(nelec),
        ecore=energy_constant,
        g_format="chemist",
        verbose=4, # adjust here if you don't want to see the detailed CCSD output.
    )
    hf_energy = ccsd_result["e_hf"]
    ccsd_energy = ccsd_result["e_tot"]
    t1 = ccsd_result["t1"]
    t2 = ccsd_result["t2"]
    mo_coeff = np.asarray(ccsd_result["mf"].mo_coeff, dtype=float)

    hcore_rhf, eri_rhf = transform_integrals_to_mo(
        hcore_site, eri_site, mo_coeff
    )

    # Inspect how well the RHF orbital basis diagonalizes the mean-field Fock operator.
    fock_site = np.asarray(ccsd_result["mf"].get_fock(), dtype=float)
    fock_rhf = mo_coeff.T @ fock_site @ mo_coeff
    fock_offdiag_norm = np.linalg.norm(fock_rhf - np.diag(np.diag(fock_rhf)))

    if run_fci:
        # Compute the FCI energy in the original site basis for reference.
        # It should be invariant under the RHF orbital rotation.
        fci_energy, _ = direct_spin1.kernel(
            hcore_site,
            eri_site,
            num_orbitals,
            nelec,
            ecore=energy_constant,
        )
    else:
        fci_energy = None

    ppp_case["hf_energy"] = hf_energy
    ppp_case["ccsd_energy"] = ccsd_energy
    ppp_case["fci_energy"] = fci_energy
    ppp_case["mo_coeff"] = mo_coeff
    ppp_case["t1"] = t1
    ppp_case["t2"] = t2
    ppp_case["hcore_rhf"] = hcore_rhf
    ppp_case["eri_rhf"] = eri_rhf
    ppp_case["fock_offdiag_norm"] = fock_offdiag_norm


classical_preprocessing(ppp_n2, run_fci=True)

print("\n=== RHF/CCSD Results ===")
print(f"RHF energy                     : {ppp_n2['hf_energy']:.6f} eV")
print(f"CCSD energy                    : {ppp_n2['ccsd_energy']:.6f} eV")
print(f"FCI energy                     : {ppp_n2['fci_energy']:.6f} eV")
print(f"t1 shape                       : {ppp_n2['t1'].shape}")
print(f"t2 shape                       : {ppp_n2['t2'].shape}")
print(f"MO coefficient matrix shape    : {ppp_n2['mo_coeff'].shape}")
print(f"||offdiag(C^T F C)||_F         : {ppp_n2['fock_offdiag_norm']:.3e}")
print(f"||h_site||_F                   : {np.linalg.norm(ppp_n2['hcore_site']):.6f}")
print(f"||h_rhf||_F                    : {np.linalg.norm(ppp_n2['hcore_rhf']):.6f}")  # should be the same as h_site due to unitary transformation



The RHF coefficient matrix `mo_coeff = C` has columns containing RHF/MO orbitals expanded in the localized **site basis**. In other words, with the convention used by PySCF,

$$
 c_\mu^\dagger = \sum_p C_{p\mu} a_p^\dagger,
$$

where $a_p^\dagger$ creates an electron in site orbital $p$ and $c_\mu^\dagger$ creates an electron in RHF orbital $\mu$. The Hamiltonian tensors in the site and RHF bases represent the same physical PPP Hamiltonian, so exact diagonalization is basis invariant. However, SQD works with finite sampled determinant subspaces, and those subspaces depend on the chosen orbital basis. This is why the problem set compares the site and RHF bases explicitly, following the tutorial.


## Problems

Now, based on the tutorial and references, solve and discuss the following problems by completing the code blocks.
Complete the following problems by referring to the SQD tutorial notebook; [web tutorial](https://quantum.cloud.ibm.com/docs/en/tutorials/sample-based-quantum-diagonalization).
Fill the code fragment marked as `TODO`.

### Problem 1. Build an LUCJ ansatz circuit from the CCSD amplitudes

**Goal.** Construct a **local unitary cluster Jastrow (LUCJ)** circuit initialized from the CCSD amplitudes obtained above.


In [ ]:
# Reference solution for Problem 1

import ffsim
from qiskit import QuantumCircuit, QuantumRegister
from qiskit_ibm_runtime.fake_provider import FakeFez, FakeNighthawk

backend_fez = FakeFez()
backend_nighthawk = FakeNighthawk()


def build_lucj_circuit(ppp_case,
                       n_reps,
                       alpha_alpha_indices,
                       alpha_beta_indices,
                       **kwargs):
    num_orbitals, nelec = ppp_case["num_orbitals"], ppp_case["nelec"]
    t1, t2 = ppp_case["t1"], ppp_case["t2"]

    ucj_op = ffsim.UCJOpSpinBalanced.from_t_amplitudes(
        # TODO: Fill the arguments
        optimize=True,
        options=dict(maxiter=10),
        **kwargs
    )    
    # TODO: Construct the circuit
    return lucj_circuit


n_reps = 1

# Heavy-hex-inspired sparse interaction pattern.
# The first list is reused for alpha-alpha and beta-beta interactions
# in the spin-balanced UCJ ansatz.
alpha_alpha_indices = [(p, p + 1) for p in range(ppp_n2["num_orbitals"] - 1)]
alpha_beta_indices = None

pass_manager, alpha_beta_indices = ffsim.qiskit.generate_lucj_pass_manager(
    # TODO: fill the argument
    interaction_pairs=(alpha_alpha_indices, None),
    optimization_level=3,
)

print(f"alpha_beta_indices = {alpha_beta_indices}")

lucj_circuit = build_lucj_circuit(ppp_n2, n_reps, alpha_alpha_indices, alpha_beta_indices)

# below code for print
# Decompose the circuit and display (the barriers are just for visual separation of the layers).
lucj_circuit_decomposed = QuantumCircuit(*lucj_circuit.qregs, *lucj_circuit.cregs)
for instr in lucj_circuit.decompose().data:
    if instr.operation.name in ["measure", "barrier"]:
        continue
    lucj_circuit_decomposed.append(instr.operation, instr.qubits, instr.clbits)
    lucj_circuit_decomposed.barrier()
display(lucj_circuit_decomposed.draw("mpl", scale=0.5, fold=-1))
# display(lucj_circuit_decomposed.decompose().draw("mpl", fold=-1))

In [ ]:
lucj_t = pass_manager.run(lucj_circuit)

print("=== LUCJ circuit summary ===")
print(f"n_reps                 : {n_reps}")
print(f"qubits                 : {lucj_t.num_qubits}")
print(f"depth                  : {lucj_t.depth()}")
print(f"size                   : {lucj_t.size()}")
print(f"same-spin pairs        : {len(alpha_alpha_indices)}")
print(f"opposite-spin pairs    : {len(alpha_beta_indices)}")
print(lucj_t.count_ops())

### Problem 2. Estimate the ground-state energy using SQD and ansatz-generated samples

**Goal.** Replace the uniformly random bitstrings from the tutorial with samples obtained from
the LUCJ ansatz, and then run SQD on those samples.

**What you should do**

1. Generate bitstring samples from the circuit built in Problem 1 using `ffsim.qiskit.FfsimSampler`.
2. Convert the raw counts to the bitstring/probability-array format used by
   `qiskit-addon-sqd`.
3. Run `diagonalize_fermionic_hamiltonian` on the **RHF-basis Hamiltonian**
   (`hcore_rhf`, `eri_rhf`), because the ansatz is defined relative to the RHF reference.
4. Use the Hartree-Fock occupations as the initial occupancy guess.
5. Track the iteration history and compare the final SQD energy against CCSD and, when
   affordable, FCI.

In [ ]:
# Reference solution for Problem 2

from functools import partial

from qiskit.primitives import StatevectorSampler
from qiskit_addon_sqd.fermion import SCIResult, diagonalize_fermionic_hamiltonian, solve_sci_batch

# ---------------------------------------------------------------------
# 1) Sample bitstrings from the LUCJ circuit.
# For a local notebook demo, Statevector sampling is simple and robust.
# For hardware execution, replace this part with Sampler / Runtime counts.
# ---------------------------------------------------------------------
shots = 200_000

def sample_bitstring_ffsimsampler(lucj_circuit,  # Untranspiled circuit
                                  ppp_case,
                                  shots=200_000,
                                  seed=1234):
    #TODO: complete the function
    unique_bitstrings = list(counts.keys())
    return bit_array, unique_bitstrings

bit_array, unique_bitstrings = sample_bitstring_ffsimsampler(lucj_circuit, ppp_n2, shots, seed=1234)

print("=== Sample summary ===")
print(f"shots                         : {shots}")
print(f"bit_array shape               : {bit_array.array.shape}")
print("number of unique bitstrings:", len(unique_bitstrings))
print(unique_bitstrings[:10])

In [ ]:
# ---------------------------------------------------------------------
# 2) Run SQD with basis-consistent samples and Hamiltonian tensors.
# ---------------------------------------------------------------------

import pandas as pd
import matplotlib.pyplot as plt

# Global SQD parameters, you may explore and adjust these as needed.
NUM_BATCHES = 1
SAMPLES_PER_BATCH = 400
ENERGY_TOL = 1e-5
OCCUPANCIES_TOL = 1e-5
MAX_ITERATIONS = 5
MAX_CYCLE = 200
CARRYOVER_THRESHOLD = 1e-6


# ---------------------------------------------------------------------
# 2) Run SQD on the RHF-basis Hamiltonian.
# ---------------------------------------------------------------------

def run_sqd_from_hardware_samples(
    ppp_case,
    bit_array,
    num_batches=NUM_BATCHES,
    samples_per_batch=SAMPLES_PER_BATCH,
    max_cycle=MAX_CYCLE,
    symmetrize_spin = True,
    carryover_threshold = CARRYOVER_THRESHOLD,
    energy_tol = ENERGY_TOL,
    occupancies_tol = OCCUPANCIES_TOL,
    max_iterations = MAX_ITERATIONS,
    seed=1234,
    **kwargs
):
    num_orbitals, nelec = ppp_case["num_orbitals"], ppp_case["nelec"]    
    energy_constant = ppp_case["energy_constant"]
    hcore_rhf, eri_rhf = ppp_case["hcore_rhf"], ppp_case["eri_rhf"]

    # TODO: Complete the SQD procedure

    return ansatz_sqd_history, ansatz_sqd_result, final_sqd_energy

sqd_history, sqd_result, final_sqd_energy = run_sqd_from_hardware_samples(ppp_n2, bit_array)

print("\n=== Energy comparison ===")
print(f"HF energy                     : {ppp_n2["hf_energy"]:.6f} eV")
print(f"CCSD energy                   : {ppp_n2["ccsd_energy"]:.6f} eV")
print(f"Final SQD energy              : {final_sqd_energy:.6f} eV")
print(f"FCI energy                    : {ppp_n2["fci_energy"]:.6f} eV")
print(f"SQD - FCI                     : {final_sqd_energy - ppp_n2["fci_energy"]:+.6f} eV")
print(f"SQD - CCSD                    : {final_sqd_energy - ppp_n2["ccsd_energy"]:+.6f} eV")


### Problem 3. Compare the heavy-hex-inspired ansatz with a Nighthawk-style ansatz in the simulator

**Goal.** Investigate how a denser ansatz changes circuit cost and SQD quality.

In Problem 1, the LUCJ ansatz used a sparse interaction pattern motivated by a heavy-hex-like layout:
same-spin interactions were nearest-neighbor along the orbital chain, while opposite-spin interactions were included only at every fourth spatial orbital.
In this problem, compare that sparse ansatz against a **Nighthawk-style ansatz**, in which same-spin interactions remain nearest-neighbor, but opposite-spin interactions are included at **every** spatial orbital.

**What you should do**

1. Reuse the sparse heavy-hex-inspired LUCJ circuit from Problem 1.
2. Build a denser Nighthawk-style ansatz with:
   - nearest-neighbor same-spin pairs (`alpha_alpha_indices`)
   $$
   (p, p+1), \qquad 0 \le p < \texttt{norb} - 1,
   $$
   - and opposite-spin pairs at every spatial orbital (`alpha_beta_indices`)
   $$
   (p, p), \qquad 0 \le p < \texttt{norb}.
   $$
3. Transpile both circuits to the same hardware-like backend and compare:
   - circuit depth,
   - circuit size,
   - the number of same-spin and opposite-spin interaction pairs.
4. Generate simulator samples from both circuits and run SQD on the RHF-basis Hamiltonian.
5. Compare the final SQD energies against CCSD and discuss the tradeoff between expressivity and circuit cost.


In [ ]:
# Reference solution for Problem 3

import pandas as pd

alpha_alpha = # TODO: determine alpha-alpha interaction indices

# From Problem 1 and Problem 2. These samples are direct RHF-basis measurements.
lucj_circuit_fez = lucj_circuit
alpha_beta_fez = alpha_beta_indices
lucj_t_fez = lucj_t
bit_array_fez, unique_bitstrings_fez = bit_array, unique_bitstrings
sqd_history_fez, sqd_result_fez, final_sqd_energy_fez = sqd_history, sqd_result, final_sqd_energy


# Build for Nighthawk
pass_manager_nighthawk, alpha_beta_nighthawk = ffsim.qiskit.generate_lucj_pass_manager(
    # TODO: Fill the argument
)

print(f"Fez alpha-beta indices: {alpha_beta_fez}")
print(f"Nighthawk alpha-beta indices: {alpha_beta_nighthawk}")

lucj_circuit_nighthawk = build_lucj_circuit(ppp_n2, n_reps, alpha_alpha, alpha_beta_nighthawk)

# Transpile for the layout/depth comparison only. ffsim sampling should use the
# untranspiled fermionic circuit, whose gates are directly supported by ffsim.
lucj_t_nighthawk = pass_manager_nighthawk.run(lucj_circuit_nighthawk)

# Execute with ffsim's sampler in the RHF/MO basis.
bit_array_nighthawk, unique_bitstrings_nighthawk, counts_nighthawk = sample_bitstring_ffsimsampler(
    lucj_circuit_nighthawk,
    ppp_n2,
    shots=shots,
    seed=1234,
)

# Postprocess with the RHF-basis Hamiltonian.
sqd_history_nighthawk, sqd_result_nighthawk, final_sqd_energy_nighthawk = run_sqd_from_hardware_samples(
    ppp_n2,
    bit_array_nighthawk,
)


In [ ]:
problem3_compare_df = pd.DataFrame(
    [
        {
            "ansatz": "Heron's heavy-hex",
            "depth": lucj_t_fez.depth(),
            "size": lucj_t_fez.size(),
            "qubits": lucj_t_fez.num_qubits,
            "same_spin_pairs": len(alpha_alpha),
            "opposite_spin_pairs": len(alpha_beta_fez),
            "unique_samples": len(unique_bitstrings_fez),
            "sqd_energy": final_sqd_energy_fez,
            "sqd_minus_ccsd": final_sqd_energy_fez - ppp_n2["ccsd_energy"],
        },
        {
            "ansatz": "Nighthawk's square",
            "depth": lucj_t_nighthawk.depth(),
            "size": lucj_t_nighthawk.size(),
            "qubits": lucj_t_nighthawk.num_qubits,
            "same_spin_pairs": len(alpha_alpha),
            "opposite_spin_pairs": len(alpha_beta_nighthawk),
            "unique_samples": len(unique_bitstrings_nighthawk),
            "sqd_energy": final_sqd_energy_nighthawk,
            "sqd_minus_ccsd": final_sqd_energy_nighthawk - ppp_n2["ccsd_energy"],
        },
    ]
)

print("\n=== Problem 3 comparison summary ===")
display(problem3_compare_df)


### Problem 4. (Advanced) Scale up to linear `n_rings = 3`, or more, but reserve sampling for a real backend

**Goal.** Integrate the workflow to scale up to the real-backend utility.

In this notebook, use the same **linear polyacene** PPP utility throughout.
In this problem, prepare the larger `n_rings = 3` case for a real backend.

**What you should do**

1. Build the PPP → RHF/CCSD → LUCJ construction workflow for the `n_rings = 3` case using the same PPP parameter setting, and compute at least:
   - the RHF energy,
   - the CCSD energy,
   - the ansatz circuit size/depth after transpilation to a hardware-like backend.
2. If you have access to a real backend, collect samples for the `n_rings = 3` circuit there and then run SQD on those hardware samples.

**Practical note.** The number of qubits will be `6 + 4 * n_rings`.


In [ ]:
ppp_n3 = prepare_case("n_rings=3", n_rings=3)
classical_preprocessing(ppp_n3, run_fci=False)

# TODO